## Preparing the dataset 

Preparation:

* Fill missing values with zeros.
* Do the split with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```

* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

In [14]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

df = pd.read_csv(data)

print(df.shape)
df.head()

(10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [15]:
df.isnull().sum()
df = df.fillna(0)

In [16]:
df.isnull().sum().sum()

np.int64(0)

In [17]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

print('train:', len(df_train))
print('validation:', len(df_val))
print('test:', len(df_test))

train: 6000
validation: 2000
test: 2000


In [18]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

df_train = df_train.drop(columns=['fuel_efficiency_mpg'])
df_val = df_val.drop(columns=['fuel_efficiency_mpg'])
df_test = df_test.drop(columns=['fuel_efficiency_mpg'])

print('X train:', df_train.shape)
print('y train:', y_train.shape)
print('target remains:', 'fuel_efficiency_mpg' in df_train.columns)

X train: (6000, 10)
y train: (6000,)
target remains: False


In [19]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print('X_train:', X_train.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test.shape)

print(list(dv.get_feature_names_out()))

X_train: (6000, 16)
X_val: (2000, 16)
X_test: (2000, 16)
['acceleration', 'drivetrain=All-wheel drive', 'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive', 'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline', 'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders', 'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA', 'vehicle_weight']


## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable. 

* Train a model with `max_depth=1`.


Which original feature is used for splitting the data? If you use
`DictVectorizer`, map a one-hot feature such as `origin=USA` back to its
original column name.


* `'vehicle_weight'`
* `'model_year'`
* `'origin'`
* `'fuel_type'`

In [12]:
from sklearn.tree import DecisionTreeRegressor, export_text

In [20]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train);

In [21]:
feature_names = list(dv.get_feature_names_out())

print(
    export_text(
        dt,
        feature_names=feature_names
    )
)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [22]:
split_feature = feature_names[dt.tree_.feature[0]]
original_feature = split_feature.split('=')[0]

print('split feature:', split_feature)
print('original feature:', original_feature)

split feature: model_year
original feature: model_year


## Question 2

Train a random forest regressor with these parameters:

* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)


What's the RMSE of this model on the validation data?

* 0.1837
* 1.837
* 18.37
* 183.7

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [24]:
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train);

In [26]:
y_pred = rf.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)

print('RMSE:', rmse)
print('Rounded:', round(rmse, 4))

RMSE: 1.8332197358745623
Rounded: 1.8332


## Question 3

Now let's experiment with the `n_estimators` parameter

* Try these values of this parameter: `[10, 50, 100, 150]`.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.


Which value gives the lowest validation RMSE? Consider 3 decimal places for
calculating the answer.

- 10
- 50
- 100
- 150

In [31]:
n_estimators_values = [10, 50, 100, 150]
results = []

for n in n_estimators_values:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    results.append((n, rmse))

    print(f'n_estimators={n:3d}  RMSE={rmse:.6f}  rounded={rmse:.3f}')

n_estimators= 10  RMSE=1.833220  rounded=1.833
n_estimators= 50  RMSE=1.773988  rounded=1.774
n_estimators=100  RMSE=1.769786  rounded=1.770
n_estimators=150  RMSE=1.769198  rounded=1.769


## Question 4

Let's select the best `max_depth`:

* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values,
  * try different values of `n_estimators` from `[10, 50, 100, 150]`
  * calculate the mean RMSE 
* Fix the random seed: `random_state=1`


What's the best `max_depth`, using the mean RMSE?

* 10
* 15
* 20
* 25

In [32]:
depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

results = []

for depth in depth_values:
    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = root_mean_squared_error(y_val, y_pred)
        results.append((depth, n, rmse))

        print(
            f'max_depth={depth:2d} '
            f'n_estimators={n:3d} '
            f'RMSE={rmse:.6f}'
        )

max_depth=10 n_estimators= 10 RMSE=1.782348
max_depth=10 n_estimators= 50 RMSE=1.747520
max_depth=10 n_estimators=100 RMSE=1.743986
max_depth=10 n_estimators=150 RMSE=1.743716
max_depth=15 n_estimators= 10 RMSE=1.831390
max_depth=15 n_estimators= 50 RMSE=1.770883
max_depth=15 n_estimators=100 RMSE=1.767872
max_depth=15 n_estimators=150 RMSE=1.766752
max_depth=20 n_estimators= 10 RMSE=1.837314
max_depth=20 n_estimators= 50 RMSE=1.772377
max_depth=20 n_estimators=100 RMSE=1.768918
max_depth=20 n_estimators=150 RMSE=1.768476
max_depth=25 n_estimators= 10 RMSE=1.836852
max_depth=25 n_estimators= 50 RMSE=1.772123
max_depth=25 n_estimators=100 RMSE=1.768320
max_depth=25 n_estimators=150 RMSE=1.767927


In [34]:
df_depth_results = pd.DataFrame(
    results,
    columns=['max_depth', 'n_estimators', 'rmse']
)

mean_rmse = (
    df_depth_results
    .groupby('max_depth')['rmse']
    .mean()
    .sort_values()
)

mean_rmse

max_depth
10    1.754392
15    1.784224
25    1.786305
20    1.786771
Name: rmse, dtype: float64

## Question 5

We can extract feature importance information from tree-based models. 

At each step of the decision tree learning algorithm, it finds the best split. 
When doing it, we can calculate "gain" - the reduction in impurity before and after the split. 
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field. 

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model


What's the most important feature (among these 4)? 

* `vehicle_weight`
*	`horsepower`
* `acceleration`
* `engine_displacement`	

In [36]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train);

In [37]:
feature_names = list(dv.get_feature_names_out())

feature_importance = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

feature_importance.head(10)

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
dtype: float64

In [38]:
options = [
    'vehicle_weight',
    'horsepower',
    'acceleration',
    'engine_displacement'
]

option_importance = (
    feature_importance
    .loc[options]
    .sort_values(ascending=False)
)

option_importance

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64

## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

* 0.3
* 0.1
* Both give equal value